# Task 4 · RGB denoising with frozen PGD
Use a T4 GPU. Run all to check the three variants on two training objects, one validation object and one test object, then train and evaluate on the complete dataset. Set RUN_FULL to False to run only the short check. The setup reuses the team's PGD environment or builds it in Colab. PGD predictions are cached in Drive and computed once per noisy cloud.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
from google.colab import drive

RUN_FULL = True
EPOCHS = 20
DATA = Path('/content/drive/MyDrive/MVP_Project/task3/gso_task3_v1')
RUN_ROOT = Path('/content/drive/MyDrive/MVP_Project/task4')
TEAM_CACHE = Path('/content/drive/MyDrive/L335_MiniProject/cache')
PGD = Path('/content/PGD')
ENV = Path('/content/mamba/envs/pgd')
PYTHON = ENV/'bin/python'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
assert 'T4' in gpu, 'Select a T4 GPU under Runtime → Change runtime type.'
drive.mount('/content/drive')
assert (DATA/'selection.json').is_file(), f'Task 3 dataset missing at {DATA}'
print('GSO data:', DATA)
print('Results root:', RUN_ROOT)

def run(args, cwd=None):
    args = list(map(str, args))
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
        if proc.wait():
            raise subprocess.CalledProcessError(proc.returncode, args)


In [ ]:
setup = 'set -euo pipefail\nexport MAMBA_ROOT_PREFIX=/content/mamba\nM=/content/bin/micromamba\ncd /content\nif [ ! -d /content/PGD/.git ]; then\n    git clone -q https://github.com/git-guocc/PGD.git /content/PGD\n    git -C /content/PGD checkout -q b5dc8be\nfi\ncase "$(git -C /content/PGD rev-parse HEAD)" in\n    b5dc8be*) ;;\n    *) echo "The PGD checkout differs from the Task 2 revision. Use a fresh runtime."; exit 1 ;;\nesac\n\nif [ ! -x "$M" ]; then\n    curl -fLs https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba\nfi\nif [ ! -x "$E/bin/python" ]; then\n    echo "Creating the PGD environment in Colab."\n    if [ -f "$CACHE/pgd-explicit.txt" ]; then\n        "$M" create -y -n pgd --file "$CACHE/pgd-explicit.txt" >/content/task4_pgd_env.log 2>&1 ||\n            { tail -60 /content/task4_pgd_env.log; exit 1; }\n    else\n        "$M" create -y -n pgd -c pytorch3d -c pytorch -c nvidia/label/cuda-11.8.0 -c nvidia -c conda-forge             python=3.10 pytorch=2.3.1 torchvision=0.18.1 "numpy<2" pytorch-cuda=11.8 pytorch3d             cuda-nvcc cuda-cudart-dev cuda-libraries-dev gcc_linux-64=11 gxx_linux-64=11 "mkl<2024.1"             >/content/task4_pgd_env.log 2>&1 || { tail -60 /content/task4_pgd_env.log; exit 1; }\n    fi\nfi\nexport PATH="$E/bin:$PATH" CUDA_HOME="$E" CONDA_PREFIX="$E" TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2\nexport CC="$E/bin/x86_64-conda-linux-gnu-gcc" CXX="$E/bin/x86_64-conda-linux-gnu-g++"\nPY="$E/bin/python"\ncd /content/PGD\nprobe=\'import torch, pytorch3d, chamfer3d, pointops_cuda; from models.pgd import PGDModel; assert torch.cuda.is_available()\'\nif ! "$PY" -c "$probe" >/content/task4_pgd_import.log 2>&1; then\n    echo "Installing PGD dependencies and CUDA extensions."\n    "$PY" -m pip install -q "numpy<2" lightning==2.4.0 pytorch-lightning==2.4.0 iopath==0.1.10         torch-geometric==2.6.1 point-cloud-utils==0.31.0 pandas tensorboard tqdm scipy\n    "$PY" -m pip install -q "numpy<2" pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv         -f https://data.pyg.org/whl/torch-2.3.1+cu118.html\n    if compgen -G "$CACHE/wheels/pgd/*.whl" >/dev/null; then\n        "$PY" -m pip install -q --no-deps "$CACHE"/wheels/pgd/*.whl\n    else\n        for ext in chamfer3d pointops; do\n            echo "Compiling $ext."\n            (cd "$ext"; "$PY" setup.py bdist_wheel -d /content/task4_pgd_wheels)                 >"/content/task4_$ext.log" 2>&1 || { tail -60 "/content/task4_$ext.log"; exit 1; }\n        done\n        "$PY" -m pip install -q --no-deps /content/task4_pgd_wheels/*.whl\n        mkdir -p "$CACHE/wheels/pgd"\n        cp /content/task4_pgd_wheels/*.whl "$CACHE/wheels/pgd/"\n    fi\nfi\n"$PY" -c "$probe"\nmkdir -p "$CACHE"\nif [ ! -f "$CACHE/pgd-explicit.txt" ]; then\n    "$M" env export -n pgd --explicit --md5 > "$CACHE/pgd-explicit.txt.tmp"\n    mv "$CACHE/pgd-explicit.txt.tmp" "$CACHE/pgd-explicit.txt"\nfi\necho \'PASS · PGD imports and CUDA ready\'\n'
setup_file = Path('/content/setup_task4_pgd.sh')
setup_file.write_text(setup)
run(['env', f'E={ENV}', f'CACHE={TEAM_CACHE}', 'bash', setup_file])


In [ ]:
script_source = '"""Train matched geometry, RGB and gated RGB residual heads on frozen PGD."""\nimport argparse\nimport csv\nfrom collections import defaultdict\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nMODES = ("geometry", "plain_rgb", "gated_rgb")\nXYZ_LEVELS = (0.01, 0.02, 0.025)\nRGB_LEVELS = (0.0, 0.05)\nPATCH_SIZE = 1000\nPATCH_OVERSAMPLING = 6\nNEIGHBORS = 16\n\n\ndef mlp(inputs, outputs):\n    return nn.Sequential(nn.Linear(inputs, 64), nn.ReLU(),\n                         nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, outputs))\n\n\nclass ResidualHead(nn.Module):\n    """Same trainable layers in all modes. Only the inputs and gate pooling differ."""\n\n    def __init__(self, mode):\n        super().__init__()\n        if mode not in MODES:\n            raise ValueError(f"Unknown model: {mode}")\n        self.mode = mode\n        self.geometry = mlp(9, 3)\n        self.fusion = mlp(15, 4)\n        for head in (self.geometry, self.fusion):\n            nn.init.zeros_(head[-1].weight)\n            nn.init.zeros_(head[-1].bias)\n\n    def forward(self, base, noisy, indices, rgb):\n        relative = base[indices] - base[:, None]\n        local_geometry = torch.cat(\n            [relative.mean(dim=1), relative.abs().mean(dim=1)], dim=-1\n        )\n        geometry = torch.cat([base - noisy, local_geometry], dim=-1)\n        if self.mode == "geometry":\n            context = local_geometry\n        else:\n            relative_rgb = rgb[indices] - rgb[:, None]\n            context = torch.cat(\n                [relative_rgb.mean(dim=1), relative_rgb.abs().mean(dim=1)], dim=-1\n            )\n\n        mixed = self.fusion(torch.cat([geometry, context], dim=-1))\n        reference = self.fusion(torch.cat([geometry, torch.zeros_like(context)], dim=-1))\n        # Isolate the change caused by context. Uniform RGB gives zero colour correction.\n        contextual_delta = mixed[:, :3] - reference[:, :3]\n        logits = mixed[:, 3:]\n        if self.mode != "gated_rgb":\n            logits = logits.mean(dim=0, keepdim=True).expand_as(logits)\n        gain = logits.sigmoid()\n        return base + self.geometry(geometry) + gain * contextual_delta, gain\n\n\n@dataclass\nclass Sample:\n    oid: str\n    split: str\n    group: str\n    noise: float\n    xyz: np.ndarray\n    clean: np.ndarray\n    colors: dict\n    metric_scale: float\n\n\ndef load_cloud(path):\n    with np.load(path, allow_pickle=False) as arrays:\n        xyz, rgb = (arrays[key].astype(np.float32) for key in ("xyz", "rgb"))\n    if (xyz.shape != rgb.shape or xyz.ndim != 2 or xyz.shape[1] != 3 or\n            not np.isfinite(xyz).all() or not np.isfinite(rgb).all() or\n            np.any(rgb < 0) or np.any(rgb > 1)):\n        raise ValueError(f"Invalid XYZRGB arrays in {path}")\n    return xyz, rgb\n\n\ndef load_samples(root, smoke):\n    assignments = json.loads((root / "selection.json").read_text())\n    split_ids = {split: sorted(oid for oid, info in assignments.items() if info["split"] == split)\n                 for split in ("train", "val", "test")}\n    if set().union(*map(set, split_ids.values())) != set(assignments):\n        raise ValueError("Unexpected split label in selection.json")\n    samples = {}\n    for split, names in split_ids.items():\n        minimum = 2 if split == "train" else 1\n        if len(names) < minimum:\n            raise ValueError(f"{split} requires at least {minimum} distinct objects")\n        samples[split] = []\n        for oid in names[:minimum] if smoke else names:\n            if Path(oid).name != oid or oid in {".", ".."}:\n                raise ValueError(f"Invalid object ID: {oid}")\n            folder = root / split / oid\n            clean, _ = load_cloud(folder / "clean.npz")\n            clean_center = (clean.max(0) + clean.min(0)) / 2\n            clean_radius = np.linalg.norm(clean - clean_center, axis=1).max()\n            if len(clean) < PATCH_SIZE or clean_radius <= 0:\n                raise ValueError(f"Invalid clean point cloud for {oid}")\n            for noise in (0.02,) if smoke else XYZ_LEVELS:\n                xyz, original = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.npz")\n                other_xyz, jitter = load_cloud(folder / f"noise_xyz_{noise:g}_rgb_0.05.npz")\n                if xyz.shape != clean.shape or not np.array_equal(xyz, other_xyz):\n                    raise ValueError(f"Point count or paired geometry mismatch for {oid} at {noise}")\n                center = (xyz.max(0) + xyz.min(0)) / 2\n                radius = np.linalg.norm(xyz - center, axis=1).max()\n                if radius <= 0:\n                    raise ValueError(f"Degenerate input cloud for {oid}")\n                samples[split].append(Sample(\n                    oid, split, assignments[oid]["group"], noise,\n                    (xyz - center) / radius, (clean - center) / radius,\n                    {"original": original, "jitter": jitter}, float(radius / clean_radius),\n                ))\n    return samples\n\n\ndef pgd_patches(model, points, knn, fps, iterations):\n    """Stitch predictions at their original point indices so each RGB stays aligned."""\n    center_count = max(1, PATCH_OVERSAMPLING * len(points) // PATCH_SIZE)\n    best = points.new_full((len(points),), float("inf"))\n    for _ in range(iterations):\n        centers, _ = fps(points[None], center_count)\n        result = torch.empty_like(points)\n        best.fill_(float("inf"))\n\n        def apply_patches(patch_centers):\n            distances, indices, neighborhoods = knn(\n                patch_centers[None], points[None], K=PATCH_SIZE, return_nn=True,\n            )\n            for center, dists, ids, patch in zip(\n                patch_centers, distances[0], indices[0], neighborhoods[0]\n            ):\n                relative = dists / dists[-1].clamp_min(1e-12)\n                predicted = model.denoise_langevin_dynamics((patch - center)[None])[0] + center\n                take = relative < best[ids]\n                result[ids[take]] = predicted[take]\n                best[ids[take]] = relative[take]\n\n        apply_patches(centers[0])\n        # FPS patches can miss a few rows. Process centred patches for any such rows.\n        for idx in torch.where(~best.isfinite())[0].tolist():\n            if not torch.isfinite(best[idx]):\n                apply_patches(points[idx:idx + 1])\n        if not best.isfinite().all() or not result.isfinite().all():\n            raise RuntimeError("PGD stitching failed to reconstruct every input row")\n        points = result\n    return points\n\n\ndef digest_file(path):\n    with path.open("rb") as stream:\n        sha = hashlib.sha256()\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef precompute(samples, pgd_root, checkpoint, iterations, cache, seed, device, knn):\n    sys.path.insert(0, str(pgd_root))\n    from models.pgd import PGDModel\n    from models.utils import farthest_point_sampling\n\n    revision = subprocess.check_output(["git", "-C", str(pgd_root), "rev-parse", "HEAD"], text=True).strip()\n    pgd_config = dict(revision=revision, weights=digest_file(checkpoint), iterations=iterations,\n                      patch_size=PATCH_SIZE, oversampling=PATCH_OVERSAMPLING, seed=seed,\n                      stitch="indexed_v1", device=torch.cuda.get_device_name(device),\n                      torch=torch.__version__)\n    model = None\n    cache.mkdir(parents=True, exist_ok=True)\n    prepared = {}\n    started = time.monotonic()\n    all_samples = [sample for split in samples.values() for sample in split]\n    for number, sample in enumerate(all_samples, 1):\n        key = hashlib.sha256(json.dumps(pgd_config, sort_keys=True).encode() + sample.xyz.tobytes())\n        filename = cache / f"{key.hexdigest()}.npy"\n        if filename.is_file():\n            baseline = np.load(filename, allow_pickle=False)\n        else:\n            if model is None:\n                model = PGDModel.load_from_checkpoint(str(checkpoint), map_location="cpu")\n                model = model.to(device).requires_grad_(False).eval()\n            # Stable result for each cache entry independently of processing order.\n            torch.manual_seed(seed)\n            with torch.no_grad():\n                baseline = pgd_patches(model, torch.from_numpy(sample.xyz).to(device), knn,\n                                       farthest_point_sampling, iterations).cpu().numpy()\n            partial = filename.with_suffix(".part")\n            with partial.open("wb") as stream:\n                np.save(stream, baseline)\n            partial.replace(filename)\n        if baseline.shape != sample.xyz.shape or not np.isfinite(baseline).all():\n            raise ValueError(f"Invalid frozen prediction for {sample.oid}")\n        xyz = torch.from_numpy(sample.xyz.copy()).to(device)\n        with torch.no_grad():\n            ids = knn(xyz[None], xyz[None], K=NEIGHBORS + 1).idx[0]\n            self_index = torch.arange(len(xyz), device=device)[:, None]\n            order = (ids == self_index).to(torch.int32).argsort(dim=1, stable=True)\n            neighbors = ids.gather(1, order[:, :NEIGHBORS]).cpu()\n        prepared[(sample.oid, sample.noise)] = (torch.from_numpy(baseline), neighbors)\n        print(f"PGD {number}/{len(all_samples)} · {sample.split}/{sample.oid} · {sample.noise:g}", flush=True)\n    print(f"PGD preparation took {time.monotonic() - started:.1f} s", flush=True)\n    return prepared, pgd_config\n\n\ndef batch(sample, prepared, condition, device):\n    base, neighbors = prepared[(sample.oid, sample.noise)]\n    return (base.to(device), torch.from_numpy(sample.xyz).to(device),\n            neighbors.to(device), torch.from_numpy(sample.colors[condition]).to(device),\n            torch.from_numpy(sample.clean).to(device))\n\n\ndef chamfer(prediction, target, knn):\n    a = knn(prediction[None], target[None], K=1).dists\n    b = knn(target[None], prediction[None], K=1).dists\n    return a.mean() + b.mean()\n\n\ndef fit(mode, samples, prepared, epochs, seed, device, knn, directory):\n    torch.manual_seed(seed)\n    model = ResidualHead(mode).to(device)\n    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    rng = np.random.default_rng(seed)\n    color_conditions = ("original", "jitter")\n    best_score, best_state = math.inf, None\n    history = []\n    for epoch in range(epochs):\n        started = time.monotonic()\n        model.train()\n        train_losses = []\n        for index in rng.permutation(len(samples["train"])):\n            sample = samples["train"][index]\n            condition = color_conditions[(epoch + index) % len(color_conditions)]\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            optim.zero_grad(set_to_none=True)\n            predicted, _ = model(base, xyz, neighbors, rgb)\n            loss = chamfer(predicted, target, knn)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Non-finite training loss for {mode} on {sample.oid}")\n            loss.backward()\n            optim.step()\n            train_losses.append(loss.item())\n        model.eval()\n        val_scores = []\n        with torch.no_grad():\n            for sample in samples["val"]:\n                for condition in color_conditions:\n                    base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n                    predicted, _ = model(base, xyz, neighbors, rgb)\n                    val_scores.append(chamfer(predicted, target, knn).item() * sample.metric_scale**2)\n        score = float(np.mean(val_scores))\n        if not math.isfinite(score):\n            raise RuntimeError(f"Non-finite validation score for {mode}")\n        history.append(dict(epoch=epoch + 1, train_cd=float(np.mean(train_losses)), validation_cd=score))\n        if score < best_score:\n            best_score = score\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n        seconds = time.monotonic() - started\n        print(f"{mode} · epoch {epoch + 1}/{epochs} · validation CD {score:.6g} · {seconds:.1f} s", flush=True)\n    torch.save(best_state, directory / f"{mode}.pt")\n    model.load_state_dict(best_state)\n    return model.eval(), history\n\n\n@torch.no_grad()\ndef evaluate(samples, heads, prepared, device, knn):\n    rows = []\n    for sample in samples:\n        for condition in ("original", "jitter"):\n            base, xyz, neighbors, rgb, target = batch(sample, prepared, condition, device)\n            predictions = {"noisy": (xyz, None), "frozen_pgd": (base, None)}\n            for mode, head in heads.items():\n                predictions[mode] = head(base, xyz, neighbors, rgb)\n            for method, (predicted, gate) in predictions.items():\n                score = chamfer(predicted, target, knn).item() * sample.metric_scale**2\n                if not math.isfinite(score):\n                    raise RuntimeError(f"Non-finite test score for {method} on {sample.oid}")\n                rows.append(dict(object_id=sample.oid, group=sample.group, xyz_noise=sample.noise,\n                                 rgb=condition, method=method, cd_squared=score,\n                                 gate_mean=gate.mean().item() if method == "gated_rgb" else ""))\n    return rows\n\n\ndef write_csv(path, rows):\n    with path.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef self_test():\n    torch.manual_seed(42)\n    xyz = torch.randn(32, 3)\n    base = xyz * 0.99\n    rgb = torch.rand_like(xyz)\n    distances = torch.cdist(xyz, xyz)\n    distances.fill_diagonal_(float("inf"))\n    indices = distances.argsort(dim=1)[:, :NEIGHBORS]\n    counts = []\n    for mode in MODES:\n        head = ResidualHead(mode)\n        counts.append(sum(p.numel() for p in head.parameters()))\n        prediction, _ = head(base, xyz, indices, rgb)\n        assert torch.equal(prediction, base)\n        optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3)\n        target = xyz * 0.97 + rgb * 0.05\n        for _ in range(4):\n            optimizer.zero_grad()\n            prediction, _ = head(base, xyz, indices, rgb)\n            (prediction - target).square().mean().backward()\n            assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in head.parameters())\n            optimizer.step()\n        prediction, gate = head(base, xyz, indices, rgb)\n        other, _ = head(base, xyz, indices, 1 - rgb)\n        assert prediction.shape == xyz.shape and torch.isfinite(prediction).all()\n        assert (0 < gate).all() and (gate < 1).all()\n        if mode == "geometry":\n            assert torch.equal(prediction, other)\n        else:\n            assert not torch.equal(prediction, other)\n            constant_a, _ = head(base, xyz, indices, torch.ones_like(rgb))\n            constant_b, _ = head(base, xyz, indices, torch.zeros_like(rgb))\n            assert torch.equal(constant_a, constant_b)\n        if mode == "gated_rgb":\n            assert gate.std() > 0\n        else:\n            assert torch.equal(gate[0].expand_as(gate), gate)\n    assert len(set(counts)) == 1\n    print(f"PASS · all three heads trained · {counts[0]} trainable parameters each")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--self-test", action="store_true")\n    parser.add_argument("--data", type=Path)\n    parser.add_argument("--pgd", type=Path, default=Path("/content/PGD"))\n    parser.add_argument("--out", type=Path)\n    parser.add_argument("--cache", type=Path)\n    parser.add_argument("--epochs", type=int, default=20)\n    parser.add_argument("--iterations", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    if args.self_test:\n        self_test()\n        return\n    if args.data is None or args.out is None or args.epochs < 1 or args.iterations < 1:\n        parser.error("A training run requires --data, --out and positive training settings")\n    if not torch.cuda.is_available():\n        parser.error("The published PGD operators require a CUDA runtime")\n    from pytorch3d.ops import knn_points\n    device = torch.device("cuda")\n    checkpoint = args.pgd / "pretrained" / "PGD.ckpt"\n    samples = load_samples(args.data, args.smoke)\n    args.out.mkdir(parents=True, exist_ok=False)\n    cache = args.cache or args.out / "cache"\n    prepared, config = precompute(samples, args.pgd.resolve(), checkpoint, args.iterations, cache,\n                                  args.seed, device, knn_points)\n    epochs = min(args.epochs, 2) if args.smoke else args.epochs\n    heads, history = {}, {}\n    for mode in MODES:\n        heads[mode], history[mode] = fit(\n            mode, samples, prepared, epochs, args.seed, device, knn_points, args.out,\n        )\n    rows = evaluate(samples["test"], heads, prepared, device, knn_points)\n    write_csv(args.out / "per_object.csv", rows)\n    groups = defaultdict(list)\n    for row in rows:\n        for group in ("all", row["group"]):\n            key = (group, row["xyz_noise"], row["rgb"], row["method"])\n            groups[key].append(row["cd_squared"])\n    write_csv(args.out / "summary.csv", [\n        dict(group=group, xyz_noise=noise, rgb=rgb, method=method, n_objects=len(values),\n             mean_cd_squared=float(np.mean(values)))\n        for (group, noise, rgb, method), values in sorted(groups.items())\n    ])\n    metadata = dict(pgd=config, epochs=epochs, seed=args.seed, smoke=args.smoke,\n                    selection_sha256=digest_file(args.data / "selection.json"), history=history,\n                    objects={split: sorted({s.oid for s in subset}) for split, subset in samples.items()})\n    (args.out / "run.json").write_text(json.dumps(metadata, indent=2) + "\\n")\n    print(f"COMPLETE · {\'smoke\' if args.smoke else \'full\'} · {len(rows)} test rows · {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n'
script = Path('/content/train_rgb_pgd.py')
script.write_text(script_source)
run([PYTHON, '-u', script, '--self-test'])


In [ ]:
import pandas as pd

def run_experiment(smoke):
    label = 'smoke' if smoke else 'full'
    out = RUN_ROOT / f'{label}_{RUN_ID}'
    args = [PYTHON, '-u', script, '--data', DATA, '--pgd', PGD, '--out', out,
            '--cache', RUN_ROOT/'pgd_cache', '--epochs', EPOCHS]
    if smoke:
        args.append('--smoke')
    run(args, cwd=PGD)
    display(pd.read_csv(out/'summary.csv').query('group == "all"'))

run_experiment(smoke=True)
if RUN_FULL:
    run_experiment(smoke=False)
